# 04 - Concept Evaluation & Semantic Alignment

In this notebook, we evaluate the **medical concepts extracted by the Sparse Autoencoder (SAE)** by comparing them with the corresponding **Open-I radiology reports**. We use **MedGemma** as a frozen external evaluator to assign an Aligned, Unaligned, or Uncertain verdict to each concept, then inspect the resulting scores and representative clinical cases.

We build on the visual embeddings from Step 01, the phrase-level semantic dictionary from Step 02, and the trained SAE from Step 03.

## 1. Setup and Imports

On Colab, clone the GitHub repository into `/content/xai-project5` to access the project scripts, requirements, and saved artifacts from Steps 01–03. An existing clone is reused when this cell is rerun. Evaluation outputs are saved to Google Drive so they survive runtime resets. Locally, use the existing repository.

Install the required libraries and authenticate with Hugging Face using `HF_TOKEN` from the environment, an existing Hugging Face login, or Colab Secrets. Colab Secrets requires the browser-based Colab UI; when unavailable (for example, from an IDE), Hugging Face may show a warning and fall back to other credentials. If no token is available, enter it in the hidden input prompt. The account must have access to the selected MedGemma model.

The clone contains the files pushed to GitHub; push local code, requirements, or artifact updates before starting a new Colab runtime.

In [4]:
import sys
import os
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo_url = "https://github.com/emmanuelmessina00/xai-project5.git"
    base_dir = "/content/xai-project5"

    if not os.path.exists(base_dir):
        subprocess.run(["git", "clone", "--depth", "1", repo_url, base_dir], check=True)
    else:
        print(f"Using existing repository: {base_dir}")

    from google.colab import drive
    drive.mount("/content/drive")
    save_dir = "/content/drive/MyDrive/xai-project5/src/results/04_evaluation"
else:
    # Support execution from the project root or src/notebooks.
    base_dir = os.getcwd() if os.path.isdir('src/scripts') else os.path.abspath(os.path.join('..', '..'))
    save_dir = os.path.join(base_dir, 'src', 'results', '04_evaluation')

feat_dir = os.path.join(base_dir, 'src', 'results', '01_feature_extraction')
dict_dir = os.path.join(base_dir, 'src', 'results', '02_dictionary_creation')
sae_dir = os.path.join(base_dir, 'src', 'results', '03_sae_training')
scripts_path = os.path.join(base_dir, 'src', 'scripts')
req_path = os.path.join(base_dir, 'requirements.txt')

for required_path in (req_path, os.path.join(scripts_path, 'sae.py')):
    if not os.path.isfile(required_path):
        raise FileNotFoundError(f"Project file missing: {required_path}. Check the repository location.")

os.makedirs(save_dir, exist_ok=True)

if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

print(f"Setup complete.\nRepository: {base_dir}\nOutput directory: {save_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete.
Repository: /content/xai-project5
Output directory: /content/drive/MyDrive/xai-project5/src/results/04_evaluation


In [5]:
!pip install -q -r "{req_path}"

In [6]:
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from IPython.display import display
from huggingface_hub import login
from transformers import AutoProcessor, AutoModelForImageTextToText

from sae import SparseAutoencoder

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [15]:
from getpass import getpass
from huggingface_hub import get_token, login

# Hugging Face handles unavailable Colab Secrets and checks saved credentials.
HF_TOKEN = os.getenv("HF_TOKEN") or get_token()

if not HF_TOKEN:
    # Hidden input also works when connected to a Colab runtime from an IDE.
    HF_TOKEN = getpass("Hugging Face token (input hidden): ").strip()

if not HF_TOKEN:
    raise ValueError("A Hugging Face token with access to MedGemma is required.")

login(token=HF_TOKEN, add_to_git_credential=False)
print("Hugging Face authentication complete.")

Hugging Face authentication complete.


## 2. Data Loading

We load the **Open-I visual embeddings and reports** extracted in Step 01. Each embedding must remain paired with its original report, since the report provides the textual evidence used during evaluation.

The semantic dictionary consists of the **phrase-level concept matrix** and its metadata from Step 02. Each row represents a UMLS synonym or an LLM-generated radiology phrase, associated with a parent medical concept.

In [9]:
dataset_path = os.path.join(feat_dir, 'biomedclip_openi_embeddings_with_reports.pt')
print(f"Loading Open-I embeddings and reports from: {dataset_path}")
dataset_data = torch.load(dataset_path, map_location=device)

vision_embeddings = dataset_data['embeddings'].to(torch.float32)
reports = dataset_data['reports']

if len(vision_embeddings) != len(reports):
    raise ValueError("Each visual embedding must have a corresponding report.")

print(f"Visual embeddings shape: {vision_embeddings.shape}")
print(f"Number of reports: {len(reports)}")

Loading Open-I embeddings and reports from: /content/xai-project5/src/results/01_feature_extraction/biomedclip_openi_embeddings_with_reports.pt
Visual embeddings shape: torch.Size([3666, 512])
Number of reports: 3666


In [10]:
matrix_path = os.path.join(dict_dir, 'biomedclip_phrase_level_concept_matrix.pt')
metadata_path = os.path.join(dict_dir, 'biomedclip_phrase_level_metadata.csv')

print(f"Loading phrase-level concept matrix from: {matrix_path}")
T_phrases = torch.load(matrix_path, map_location=device)
phrase_metadata = pd.read_csv(metadata_path)

if len(T_phrases) != len(phrase_metadata):
    raise ValueError("Each phrase embedding must have a corresponding metadata row.")

print(f"Phrase-level matrix shape: {T_phrases.shape}")
print(f"Number of parent concepts: {phrase_metadata['parent_concept'].nunique()}")

Loading phrase-level concept matrix from: /content/xai-project5/src/results/02_dictionary_creation/biomedclip_phrase_level_concept_matrix.pt
Phrase-level matrix shape: torch.Size([998, 512])
Number of parent concepts: 100


## 3. Sparse Autoencoder Loading & Semantic Grounding

We load the same **best SAE checkpoint** selected for semantic grounding in Step 03. The SAE remains fixed throughout evaluation.

This copy uses the current phrase-level dictionary and checkpoint from Steps 02-03. The original evaluation notebook used an older concept dictionary and SAE checkpoint, so its saved numerical results should not be treated as results for this configuration.

In [11]:
INPUT_DIM = vision_embeddings.shape[1]
HIDDEN_DIM = 1024
selected_model = 'sae_model_best_e1000_l0.002_h1024.pt'

sae = SparseAutoencoder(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM).to(device)
sae_path = os.path.join(sae_dir, selected_model)
print(f"Loading SAE weights from: {sae_path}")
state_dict = torch.load(sae_path, map_location=device)

if isinstance(state_dict, dict) and 'state_dict' in state_dict:
    sae.load_state_dict(state_dict['state_dict'])
else:
    sae.load_state_dict(state_dict)

sae.eval()
print("SAE loaded and ready for evaluation.")


Loading SAE weights from: /content/xai-project5/src/results/03_sae_training/sae_model_best_e1000_l0.05_h1024.pt
SAE loaded and ready for evaluation.


### Phrase-to-Concept Aggregation

Following Step 03, we compute the similarity between each normalized text embedding and each normalized SAE decoder column:

$$S_{\mathrm{phrases}} = T_{\mathrm{phrases}} W_{\mathrm{dec}}.$$

We use the full **998 fine-grained phrase-level concept dictionary** ($S_{\mathrm{phrases}} \in \mathbb{R}^{998 \times 1024}$). For each active neuron, the highest-scoring fine-grained phrase concept becomes its semantic label, providing a near 1-to-1 matching ($998 \approx 1024$) with the SAE hidden neurons.


In [12]:
with torch.no_grad():
    sae_decoder_weights = F.normalize(sae.decoder.weight.data, p=2, dim=0)
    phrase_similarities = torch.matmul(T_phrases, sae_decoder_weights)

# Use full 998 fine-grained phrase-level concept dictionary (near 1-to-1 matching with 1024 SAE neurons)
similarities = phrase_similarities
medical_concepts = phrase_metadata['text'].tolist()

print(f"Phrase-level concept similarities matrix shape: {similarities.shape}")
print(f"Total fine-grained medical phrase concepts: {len(medical_concepts)}")


Concept similarities matrix shape: torch.Size([100, 1024])


## 4. Frozen External Evaluator: MedGemma

We use **MedGemma** (`google/medgemma-1.5-4b-it`) to compare each extracted concept with its paired radiology report. Although we load the model through its multimodal interface, this evaluation uses **text only**: the evaluator receives the report and a proposed concept, without the image or SAE activations.

The evaluator is not trained or updated. We retain the original few-shot prompt and generation settings so that the evaluation protocol remains consistent with the original notebook. A GPU is recommended for this inference step.

In [18]:
model_id = "google/medgemma-1.5-4b-it"

model_eval = AutoModelForImageTextToText.from_pretrained(
    model_id,
    dtype=torch.bfloat16,   # Use the same precision as the original evaluation.
    device_map="auto",
)
processor_eval = AutoProcessor.from_pretrained(model_id)

model_eval.eval()
model_eval.requires_grad_(False)
print("MedGemma loaded and ready as a frozen external evaluator.")

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

MedGemma loaded and ready as a frozen external evaluator.


### Report-Concept Classification

For each concept, the evaluator produces one of three labels:

1. **Aligned**: the proposed concept agrees with the report.
2. **Unaligned**: the proposed concept contradicts the report.
3. **Uncertain**: the relationship is ambiguous or cannot be established from the report.

We configure MedGemma to perform **1-token deterministic greedy decoding** (`max_new_tokens=5`, `do_sample=False`). Reports are preprocessed to remove anonymization artifacts (`XXXX`). MedGemma outputs a single deterministic verdict (`Aligned`, `Unaligned`, or `Uncertain`), eliminating false-uncertain fallbacks caused by truncated text explanations.


In [19]:
import re

def clean_report(text):
    if not text or not isinstance(text, str):
        return ""
    # Remove repeated XXXX patterns or masked dates (XXXX-XX-XX)
    cleaned = re.sub(r'(?:XXXX|\d{4}-\d{2}-\d{2})\b\s*', '', text)
    # Remove redundant whitespace
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned


def evaluate_concept_with_llm(report, concept, model, processor, device):
    # Clean report text from anonymization artifacts (XXXX)
    cleaned_rep = clean_report(report)

    prompt_text = f"""You are a text-only medical NLP classifier. You are NOT given any image, and none is needed — this task is based purely on the written report text below.
Do not ask for clarification, do not mention missing information, do not request an image. Just classify.

Task: Determine if the Concept is Aligned, Unaligned, or Uncertain with respect to the Report text.
Rules:
- Aligned: The concept is explicitly supported or mentioned as present in the Report.
- Unaligned: The concept is explicitly negated (e.g. "no pneumothorax") or directly contradicts the Report.
- Uncertain: The report provides no mention or evidence regarding the concept.

Reply with EXACTLY ONE WORD: Aligned, Unaligned, or Uncertain.

--- EXAMPLES ---
Report: "The cardiac silhouette and mediastinum size are within normal limits. Normal chest x-ray."
Concept: "airspace disease"
Verdict: Unaligned

Report: "Lungs are overall hyperexpanded with flattening of the diaphragms. Degenerative changes in the thoracic spine."
Concept: "bone diseases"
Verdict: Aligned

Report: "Heart size is normal. Lungs are clear bilaterally. No acute cardiopulmonary abnormality."
Concept: "aortic aneurysm"
Verdict: Unaligned

Report: "Opacity in the right lower lobe, cannot exclude pneumonia."
Concept: "blister"
Verdict: Uncertain
--- END OF EXAMPLES ---

Now evaluate this real case (text only, no image involved):
Report: "{cleaned_rep}"
Concept: "{concept}"
Verdict:"""

    messages = [{"role": "user", "content": [{"type": "text", "text": prompt_text}]}]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generation = model.generate(
            **inputs,
            min_new_tokens=1,
            max_new_tokens=5,
            do_sample=False,
        )
        generation = generation[0][input_len:]

    generated_text = processor.decode(generation, skip_special_tokens=True).strip()
    text_lower = generated_text.lower()

    if "unaligned" in text_lower:
        verdict = "Unaligned"
    elif "aligned" in text_lower:
        verdict = "Aligned"
    elif "uncertain" in text_lower:
        verdict = "Uncertain"
    else:
        verdict = "Uncertain"

    print(f"[DEBUG] {concept!r} -> {generated_text!r} => {verdict}")
    return verdict


## 5. Concept Extraction & Quantitative Evaluation

For each selected Open-I embedding, we obtain the SAE activations and retain up to **TOP_K = 10 neurons** meeting an activation threshold ($z_j > 0.01$). We map these neurons to their closest phrase concepts with cosine similarity $\ge 0.20$ and remove duplicate concept labels before querying MedGemma.

For an image $i$ with a non-empty set of predicted concepts $C_i$, each score is the fraction of concepts assigned the corresponding verdict:

$$\mathrm{Score}_{v}(i) = \frac{1}{|C_i|}\sum_{c \in C_i}\mathbb{1}[\mathrm{verdict}(c, r_i)=v],$$

where $v$ is Aligned, Unaligned, or Uncertain, and $r_i$ is the associated report. The three scores therefore sum to one for each evaluated image.

As in the original notebook, we inspect the **first 10 dataset entries**. Entries without a usable report or positively activated neurons are skipped, so the final number of evaluated images may be smaller than `NUM_SAMPLES`.


In [20]:
TOP_K = 10
NUM_SAMPLES = 10

In [21]:
results = []

print(f"Evaluating up to {NUM_SAMPLES} images using MedGemma...\n")

for i in tqdm(range(min(NUM_SAMPLES, len(vision_embeddings)))):
    img_emb = vision_embeddings[i].unsqueeze(0).to(device)

    try:
        report = reports[i]
    except Exception:
        continue

    cleaned_rep = clean_report(report)

    if not cleaned_rep or len(cleaned_rep) < 5:
        continue

    with torch.no_grad():
        _, z = sae(img_emb)

    activations = z[0]

    top_vals, top_indices = torch.topk(activations, TOP_K)
    # Threshold positive activations z_j > 0.01 to filter out noise
    active_mask = top_vals > 0.01
    active_neurons = top_indices[active_mask]

    if len(active_neurons) == 0:
        continue

    predicted_concepts = set()

    for neuron_idx in active_neurons:
        neuron_concept_scores = similarities[:, neuron_idx]
        best_val, best_concept_idx = torch.max(neuron_concept_scores, dim=0)
        # Filter weak cosine similarities (< 0.20)
        if best_val.item() >= 0.20:
            concept = medical_concepts[best_concept_idx.item()]
            predicted_concepts.add(concept)

    predicted_concepts = sorted(predicted_concepts)

    aligned_count = 0
    unaligned_count = 0
    uncertain_count = 0
    concept_details = []

    for concept in predicted_concepts:
        verdict = evaluate_concept_with_llm(cleaned_rep, concept, model_eval, processor_eval, device)

        if verdict == "Aligned":
            aligned_count += 1
        elif verdict == "Unaligned":
            unaligned_count += 1
        else:
            uncertain_count += 1

        concept_details.append({"concept": concept, "verdict": verdict})

    total = len(predicted_concepts)

    if total > 0:
        results.append({
            "image_index": i,
            "report": cleaned_rep,
            "total_concepts_found": total,
            "aligned_score": aligned_count / total,
            "unaligned_score": unaligned_count / total,
            "uncertain_score": uncertain_count / total,
            "details": concept_details
        })


Evaluating up to 10 images using MedGemma...



  0%|          | 0/10 [00:00<?, ?it/s]

[DEBUG] 'atherosclerosis' -> tail='...Uncertain' => Uncertain
[DEBUG] 'bone sclerosis' -> tail='...Uncertain' => Uncertain
[DEBUG] 'cardiomegaly' -> tail='...Uncertain' => Uncertain
[DEBUG] 'heart atria' -> tail='...Uncertain' => Uncertain
[DEBUG] 'heart ventricles' -> tail='...Uncertain' => Uncertain
[DEBUG] 'kyphosis' -> tail='...Uncertain' => Uncertain
[DEBUG] 'pectus carinatum' -> tail='...Uncertain' => Uncertain


 10%|█         | 1/10 [00:03<00:27,  3.04s/it]

[DEBUG] 'shoulder' -> tail='...Uncertain' => Uncertain
[DEBUG] 'bone and bones' -> tail='...Uncertain' => Uncertain
[DEBUG] 'bone fracture' -> tail='...   *  "Midline sternotomy XXXXX." - This indicates surgery involving the breastbone (sternum). Stern' => Uncertain
[DEBUG] 'bone sclerosis' -> tail='...  *  "Midline sternotomy XXXXX." - Surgical scar/incision site. Not related to bone structure itself' => Uncertain
[DEBUG] 'cardiac silhouette' -> tail='...ul monary arte ries. Cle ar lungs. In feri or XXXX XXX X XXXX. N o ac ute pulmon ary find ings."

3.' => Uncertain
[DEBUG] 'cardiomegaly' -> tail='...omegaly means enlarged heart.
    *  "...Midline sternotomy ..." - Surgical procedure related to the' => Uncertain
[DEBUG] 'humerus' -> tail='...Uncertain' => Uncertain
[DEBUG] 'pericardial effusion' -> tail='...Uncertain' => Uncertain
[DEBUG] 'pulmonary fibrosis' -> tail='...Uncertain' => Uncertain


...

[DEBUG] 'trachea' -> tail='...**no focal air space opacit y** to suggest a pne umonia. There is ... cervical spinal fusion..."

3.' => Uncertain
[DEBUG] 'atherosclerosis' -> tail='...  "cardiac silhouette is not [enlarged]": This might suggest something else causing enlargement, but' => Uncertain
[DEBUG] 'calcified granuloma' -> tail='...is "*granulomas*". This difference might be relevant, but let's focus first on whether calcification' => Uncertain
[DEBUG] 'cystic fibrosis' -> tail='...uette...not enlarged": Normal finding.
    -"apparent interval increase in **low density convexity**' => Uncertain
[DEBUG] 'dislocations' -> tail='...The cardiac silhouette is **not enlarged.**" - Rules out some causes but doesn't confirm dislocation' => Uncertain
[DEBUG] 'expansile bone lesions' -> tail='..." "widening," "lucent lesion," "bone destruction," etc., specifically in relation to ribs, vertebrae' => Uncertain
[DEBUG] 'pulmonary mass' -> tail='...  *  "The cardiac silhouette is **not enlarge

 90%|█████████ | 9/10 [06:37<00:52, 52.53s/it]

[DEBUG] 'tuberculosis' -> tail='... -   "cardiac silhouette...not enlarged": Negative finding regarding heart size. Not directly TB but' => Uncertain
[DEBUG] 'dislocations' -> tail='... normal limits for si... [cut short]" - This part talks about heart/mediastinum size and shape being' => Uncertain
[DEBUG] 'heart failure' -> tail='...thin normal limits for si... [cut short]" - This part seems like it might be about heart/mediastinum' => Uncertain
[DEBUG] 'kyphosis' -> tail='... normal limits for si... [cut short]" - This sentence describes the heart/mediastinum size and shape' => Uncertain
[DEBUG] 'pneumothorax' -> tail='... normal limits for si... [cut short]" - This sentence describes the heart/mediastinum size and shape' => Uncertain
[DEBUG] 'pulmonary infiltrate' -> tail='...mal limits for si... [cut short]" - This part seems irrelevant to the core findings about the lungs/' => Uncertain
[DEBUG] 'sarcoidosis' -> tail='...Uncertain' => Uncertain
[DEBUG] 'trachea' -> tail='...tinal s

100%|██████████| 10/10 [07:46<00:00, 46.69s/it]

[DEBUG] 'tuberculosis' -> tail='...o lungs exist normally inflated out evidence about focal airspace sick, pleural effusionity, instead' => Uncertain


### Evaluation Results

We display the per-image results and the mean Aligned, Unaligned, and Uncertain scores across evaluated images. These means give each image equal weight, regardless of the number of distinct extracted concepts.

In [22]:
result_columns = [
    'image_index', 'report', 'total_concepts_found',
    'aligned_score', 'unaligned_score', 'uncertain_score', 'details'
]
df_evaluation = pd.DataFrame(results, columns=result_columns)

print(f"Evaluation complete. Evaluated {len(df_evaluation)} valid images.")
display(df_evaluation.head())

if not df_evaluation.empty:
    score_columns = ['aligned_score', 'unaligned_score', 'uncertain_score']
    display(df_evaluation[score_columns].mean().to_frame(name='Mean score'))
else:
    print("No valid images were evaluated. Check the reports and SAE activations.")

Evaluation complete. Evaluated 10 valid images.


,image_index,report,total_concepts_found,aligned_score,unaligned_score,uncertain_score,details
0,0,The cardiac silhouette and mediastinum size ar...,8,0.000000,0.0,1.000000,"[{'concept': 'atherosclerosis', 'verdict': 'Un..."
1,1,Borderline cardiomegaly. Midline sternotomy XX...,9,0.000000,0.0,1.000000,"[{'concept': 'bone and bones', 'verdict': 'Unc..."
2,2,"No displaced rib fractures, pneumothorax, or p...",9,0.000000,0.0,1.000000,"[{'concept': 'bone diseases, metabolic', 'verd..."
3,3,There are diffuse bilateral interstitial and a...,7,0.142857,0.0,0.857143,"[{'concept': 'cystic fibrosis', 'verdict': 'Un..."
4,4,The cardiomediastinal silhouette and pulmonary...,10,0.000000,0.0,1.000000,"[{'concept': 'aorta, thoracic', 'verdict': 'Un..."


,Mean score
aligned_score,0.014286
unaligned_score,0.000000
uncertain_score,0.985714


## 6. Qualitative Analysis: Best, Median & Worst Cases

We rank the evaluated images by their **Aligned score** and inspect the first, middle, and last rows. For each case, we display the original report, the three scores, and the concept-level verdicts.

These cases help us examine agreement, ambiguity, and possible mismatches. The ranking is based only on the Aligned score: a low score can reflect uncertainty as well as contradiction. Tied scores do not establish a meaningful difference in explanation quality, and with fewer than three evaluated images, a case can appear more than once.

In [23]:
valid_df = df_evaluation[df_evaluation['total_concepts_found'] > 0].copy()
sorted_df = valid_df.sort_values(by='aligned_score', ascending=False).reset_index(drop=True)

if sorted_df.empty:
    print("No evaluated cases are available for qualitative analysis.")
else:
    case_studies = {
        "BEST CASE (Highest Aligned Score)": sorted_df.iloc[0],
        "MEDIAN CASE (Middle-Ranked Image)": sorted_df.iloc[len(sorted_df) // 2],
        "WORST CASE (Lowest Aligned Score)": sorted_df.iloc[-1]
    }

    for title, case in case_studies.items():
        print("-" * 60)
        print(title)
        print(f"Original image index: {case['image_index']}")
        print(
            f"Scores -> Aligned: {case['aligned_score']:.2f} | "
            f"Unaligned: {case['unaligned_score']:.2f} | "
            f"Uncertain: {case['uncertain_score']:.2f}"
        )
        print(f"\nORIGINAL RADIOLOGY REPORT:\n{case['report']}\n")
        print("SAE CONCEPTS AND MEDGEMMA VERDICTS:")
        for detail in case['details']:
            print(f"  {detail['concept']:<25} -> {detail['verdict']}")
        print()

------------------------------------------------------------
BEST CASE (Highest Aligned Score)
Original image index: 3
Scores -> Aligned: 0.14 | Unaligned: 0.00 | Uncertain: 0.86

ORIGINAL RADIOLOGY REPORT:
There are diffuse bilateral interstitial and alveolar opacities consistent with chronic obstructive lung disease and bullous emphysema. There are irregular opacities in the left lung apex, that could represent a cavitary lesion in the left lung apex.There are streaky opacities in the right upper lobe, XXXX scarring. The cardiomediastinal silhouette is normal in size and contour. There is no pneumothorax or large pleural effusion. 1. Bullous emphysema and interstitial fibrosis. 2. Probably scarring in the left apex, although difficult to exclude a cavitary lesion. 3. Opacities in the bilateral upper lobes could represent scarring, however the absence of comparison exam, recommend short interval followup radiograph or CT thorax to document resolution.

SAE CONCEPTS AND MEDGEMMA VERDIC

### Interpretation of the Scores

These scores measure **agreement with the written report under the chosen evaluation protocol**. They do not directly establish whether a concept is visible in the image or whether the SAE representation is clinically correct. Reports may omit findings, and the evaluator's prompt and generated responses can affect the verdicts.

The 10-entry subset provides an initial inspection of the pipeline. Broader evaluation and manual review of ambiguous or inconsistent cases are needed before drawing general conclusions about concept quality.